# 🦜 LangChain Introduction

## Learning Objectives
In this notebook, you will learn:
1. **What LangChain is** - a framework for composing LLM calls into pipelines
2. **LCEL basics** - chaining a prompt, an LLM, and an output parser with the `|` operator
3. **Multi-step pipelines** - combining chains so one chain's output feeds the next (detail → summary)
4. **Extending a pipeline** - appending a third step (sentiment analysis) the same way
5. **An assistant loop** - wrapping a chain in a `while` loop that takes one topic per turn

## Where this fits
This is the first notebook of the course. It shows the shape of every LangChain program —
prompt, model, parser, piped together — before the next notebooks compare model providers.

## Prerequisites
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

Load the API key from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: one chat model shared by every chain in this notebook
# ============================================================================
from aied_helpers import get_openai_llm

# temperature=0.7 gives slightly varied, more natural explanations
llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🦜 Part 1: What LangChain is

LangChain wraps every chat model in the same interface. `llm.invoke(...)` sends one prompt and
returns an `AIMessage`; the text is on its `.content` attribute. Everything else in LangChain —
prompts, parsers, chains — composes around that one call.

In [ ]:
# ============================================================================
# FIRST MODEL CALL: invoke the chat model directly
# ============================================================================
response = llm.invoke("What is the capital of France?")
print(response.content)

print("-------------------------------------------------")

alt_response = llm.invoke("What is LangChain? Explain in 1 line")
print(alt_response.content)

---
## ⛓️ Part 2: Your first LangChain pipeline

A LangChain pipeline is a sequence of steps where inputs are processed to produce outputs.
Build a pipeline that:
* Accepts a topic as input.
* Uses the LLM to generate a response.
* Outputs the generated text as a plain string.

### Key Concepts:
- **`PromptTemplate`**: a prompt with `{placeholders}` filled at invoke time
- **`StrOutputParser`**: turns the model's `AIMessage` into a plain string
- **`|` (LCEL)**: pipes the output of one step into the next

In [ ]:
# ============================================================================
# FIRST PIPELINE: prompt -> llm -> output parser
# ============================================================================
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

# Step 1: Create a Prompt Template
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Provide me details about {topic}?",
)

# Step 2: Create a Chain using LCEL (LangChain Expression Language)
# The | operator chains: prompt -> llm -> output parser
chain = prompt | llm | StrOutputParser()

# Step 3: Generate the responses
response = chain.invoke({"topic": "Generative AI"})
resp = chain.invoke({"topic": "Deep Learning"})

print(response)
print("-------------------------------------------------")
print(resp)

---
## 🔗 Part 3: A multi-step pipeline

Modify the project: after generating the details, summarize them into one sentence. The
output of the first chain becomes the input of the second.

### Key Concepts:
- **`RunnablePassthrough`**: passes the raw input string through so it can fill `{topic}`
- **A lambda in a chain**: reshapes one step's string output into the dict the next prompt needs

In [ ]:
# ============================================================================
# MULTI-STEP PIPELINE: details -> one-sentence summary
# ============================================================================
from langchain_core.runnables import RunnablePassthrough

# Step 1: Create Prompt Templates
detail_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Provide me details about {topic}?",
)

summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="Summarize the following text into a single concise sentence: {text}",
)

# Step 2: Create individual chains using LCEL
answer_chain = detail_prompt | llm | StrOutputParser()
summary_chain = summary_prompt | llm | StrOutputParser()

# Step 3: Combine the chains: the first generates details, then passes them to the summary
pipeline = (
    {"topic": RunnablePassthrough()}
    | answer_chain
    | (lambda text: {"text": text})
    | summary_chain
)

# Step 4: Generate the response
response = pipeline.invoke("Deep Learning")
print(response)

---
## 😊 Part 4: Add a sentiment-analysis step

A third stage is appended exactly like the second: reshape the summary into `{"text": ...}`
and pipe it into a sentiment chain.

In [ ]:
# ============================================================================
# EXTENDED PIPELINE: details -> summary -> sentiment
# ============================================================================
sentiment_prompt = PromptTemplate(
    input_variables=["text"],
    template="Analyze the sentiment of the following text: {text}",
)

sentiment_chain = sentiment_prompt | llm | StrOutputParser()

full_pipeline = (
    {"topic": RunnablePassthrough()}
    | answer_chain
    | (lambda text: {"text": text})
    | summary_chain
    | (lambda text: {"text": text})
    | sentiment_chain
)

response = full_pipeline.invoke("Deep Learning")
print(response)

---
## 💬 Part 5: An assistant loop

A `while` loop around `answer_chain.invoke(...)` turns any chain into an assistant. By default
the loop reads topics from a short scripted list so the notebook runs top to bottom unattended.
Set `INTERACTIVE = True` to type your own topics (type `exit` to quit).

In [ ]:
# ============================================================================
# ASSISTANT LOOP: one chain call per topic until 'exit'
# ============================================================================
INTERACTIVE = False  # True -> type topics with input()
scripted_topics = iter(["Vector databases", "exit"])

while True:
    if INTERACTIVE:
        user_input = input("Enter the topic you want to know more about (or type 'exit' to quit): ")
    else:
        user_input = next(scripted_topics)
        print(f"📝 Topic: {user_input}")

    # Exit the loop if the user types 'exit'
    if user_input.lower() == "exit":
        print("Exiting the assistant. GoodBye!!")
        break

    print(answer_chain.invoke({"topic": user_input}))

---
## 📝 Summary

In this notebook, we learned:

### 1. What LangChain is
- `llm.invoke(...)` sends a single prompt to a chat model and returns a message with a `.content` attribute

### 2. LCEL basics
- LCEL's `|` operator chains a `PromptTemplate`, an LLM, and an output parser into one runnable

### 3. Multi-step pipelines
- Chains compose by piping one chain's output into the next (detail chain → summary chain)
- `RunnablePassthrough` lets a raw string input flow unchanged into the first step

### 4. Extending a pipeline
- A third stage (sentiment analysis) is appended the same way: detail → summary → sentiment

### 5. An assistant loop
- A `while` loop around `chain.invoke(...)` turns any chain into an assistant; a scripted input list keeps it runnable unattended

### Next Steps
- **`02_Commercial_LLMs_Natively.ipynb`**: call commercial LLM providers directly, without LangChain